# Notebook 06 — Delivery Logistics & Customer Experience (CSAT) Deep-Dive

**Project:** Enterprise E-Commerce Operations and Customer Experience Control Tower  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Phase:** 7 — Exploratory Data Analysis & Operational Intelligence  

## Core Operational Insights Covered
1. **Delivery Lead-Time & SLA Fulfillment:** Promised vs actual delivery duration, on-time delivery rate (91.89%).
2. **Operational Duration Split:** Seller handling vs carrier transit times.
3. **Geographic Distance Degradation:** Haversine distance bands and freight cost escalation.
4. **Customer Experience & CSAT Drivers:** Direct impact of delivery delay severity on customer review ratings.

---
## 0. Setup & Data Loading

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

DB_PATH = '../data/processed/ecommerce_control_tower.db'
conn = sqlite3.connect(DB_PATH)

fact_orders = pd.read_sql_query('SELECT * FROM fact_orders WHERE order_status = "delivered";', conn)
fact_items = pd.read_sql_query('SELECT * FROM fact_order_items WHERE order_status = "delivered";', conn)

print(f'Loaded {len(fact_orders):,} delivered orders and {len(fact_items):,} delivered items.')

---
## 1. Delivery SLA & Delay Distribution

In [ ]:
# Delivery duration statistics
deliv_stats = fact_orders[['delivery_days', 'delay_days', 'handling_days', 'transit_days', 'shipping_sla_days']].describe()
print('Delivery Lead Time & SLA Metrics Summary (Days):')
display(deliv_stats)

In [ ]:
# Delivery Delay Distribution (Actual vs Promised)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(fact_orders['delay_days'].clip(lower=-25, upper=25), bins=50, color='#2b5c8f', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Promised SLA Deadline (0 Days)')
ax.axvline(fact_orders['delay_days'].median(), color='green', linestyle=':', linewidth=2, label=f'Median: {fact_orders["delay_days"].median():.1f} days early')
ax.set_title('Distribution of Delivery Delay Variance (Days Early / Late)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Delay Days (Negative = Delivered Ahead of Schedule, Positive = Late)')
ax.set_ylabel('Delivered Orders Count')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Customer CSAT vs Delivery Delay Severity

In [ ]:
# CSAT Degradation by Delay Band
csat_bands = fact_orders[fact_orders['review_score_avg'].notna()].groupby('delay_band').agg(
    orders=('order_id', 'count'),
    avg_score=('review_score_avg', 'mean'),
    detractor_pct=('low_review_flag', lambda x: np.mean(x) * 100),
    promoter_pct=('high_review_flag', lambda x: np.mean(x) * 100)
).reindex(['Early or On Time', '1–3 Days Late', '4–7 Days Late', '8+ Days Late']).reset_index()

print('Customer Experience & CSAT by Delivery SLA Band:')
display(csat_bands)

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(csat_bands['delay_band'], csat_bands['avg_score'], color=['#2ca02c', '#ffbb78', '#ff7f0e', '#d62728'], width=0.5)
for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, y + 0.08, f'{y:.2f}', ha='center', fontweight='bold')
ax.set_ylim(0, 5.0)
ax.set_title('Direct CSAT Impact of Delivery Delay Severity', fontsize=13, fontweight='bold', pad=15)
ax.set_ylabel('Average Review Score (1–5 Stars)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Geospatial Shipping Distance & Freight Friction

In [ ]:
# Logistics degradation across Haversine distance tiers
dist_df = fact_items.groupby('distance_band').agg(
    items=('order_item_id', 'count'),
    avg_delivery_days=('delivery_days', 'mean'),
    avg_freight=('freight_value', 'mean'),
    late_rate=('late_delivery_flag', lambda x: np.mean(x) * 100)
).reindex(['0–100 km', '101–500 km', '501–1,000 km', '1,001–2,000 km', '2,000+ km']).reset_index()

print('Distance Band Logistics Performance:')
display(dist_df)